# SCF Phase 2 shard 07

Sweeps: nullcal, power. Jobs: 23. Projected: 4.6 h
(safety-factor 1.8x applied). Grids hash: `68970a975545`.
Code source: github.com/hugogobato/scf-confounding-frontier @ tag `phase2-freeze` (pinned for
reproducibility).
Pre-registration: `docs/phase2_preregistration.md` (thresholds frozen before
any data generation; deviation register D1-D7 included there).

Resume-safe: completed cells are skipped on rerun (checkpoint parquet per
cell). If the notebook approaches the Colab wall limit it finishes the
current cell and stops cleanly; rerun to continue.

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"
!pip install -q "numpy>=2.0" "scipy>=1.14" "pandas>=2.2" "pyarrow>=16" scikit-learn

In [ ]:
!git clone --depth 1 --branch phase2-freeze \
    https://github.com/hugogobato/scf-confounding-frontier.git scf_repo
import sys, hashlib, json
sys.path.insert(0, "scf_repo/code")
# verify the pinned code matches the manifest recorded at generation time
EXPECTED = json.loads("{\"de_formulas.py\": \"5dffb441b638\", \"simulator.py\": \"ef31ca2a201b\", \"estimators.py\": \"7e27f25b2330\", \"detection.py\": \"06586fe60b9f\", \"runners.py\": \"df67486b60f5\"}")
for fname, short in EXPECTED.items():
    h = hashlib.sha256(open(f"scf_repo/code/{fname}", "rb").read()).hexdigest()[:12]
    assert h == short, f"code mismatch: {fname} ({h} != {short})"
print("code verified against generation-time hashes")

In [ ]:
import json, time, traceback
from multiprocessing import Pool
from runners import run_cell

JOBS = json.loads("[{\"config\": {\"n\": 2000, \"p\": 10000, \"r\": 3, \"l\": [1.118033988749895, 1.118033988749895, 1.118033988749895], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"g\": 0.0, \"twin_gamma0\": false, \"q_fixed\": true}, \"config_id\": \"fb35861fe8e9\", \"mode\": \"nullcal\", \"sweep\": \"nullcal\", \"reps\": 1200, \"raw_path\": \"data/sim/nullcal/raw/fb35861fe8e9.parquet\", \"means_path\": \"data/sim/nullcal/means/fb35861fe8e9.npz\", \"_rank\": 1, \"_sweep\": \"nullcal\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 3, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 1.6, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"5d8f27ee695e\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/5d8f27ee695e.parquet\", \"means_path\": \"data/sim/power/means/5d8f27ee695e.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 3, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 3.2, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"68c756e0e09c\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/68c756e0e09c.parquet\", \"means_path\": \"data/sim/power/means/68c756e0e09c.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"g\": 0.15, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"0084f1f38a6e\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/0084f1f38a6e.parquet\", \"means_path\": \"data/sim/power/means/0084f1f38a6e.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"g\": 0.4, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"f0519275af40\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/f0519275af40.parquet\", \"means_path\": \"data/sim/power/means/f0519275af40.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"g\": 0.8, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"95e109c3fc8d\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/95e109c3fc8d.parquet\", \"means_path\": \"data/sim/power/means/95e109c3fc8d.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"g\": 1.6, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"a2ab8f26ad73\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/a2ab8f26ad73.parquet\", \"means_path\": \"data/sim/power/means/a2ab8f26ad73.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"g\": 3.2, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"118ac9c82299\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/118ac9c82299.parquet\", \"means_path\": \"data/sim/power/means/118ac9c82299.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"g\": 0.15, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"aa86fb25967d\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/aa86fb25967d.parquet\", \"means_path\": \"data/sim/power/means/aa86fb25967d.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"g\": 0.4, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"bf4ef63c368b\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/bf4ef63c368b.parquet\", \"means_path\": \"data/sim/power/means/bf4ef63c368b.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"g\": 0.8, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"4a4db7951b96\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/4a4db7951b96.parquet\", \"means_path\": \"data/sim/power/means/4a4db7951b96.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"g\": 1.6, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"86ea293b9a1e\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/86ea293b9a1e.parquet\", \"means_path\": \"data/sim/power/means/86ea293b9a1e.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"g\": 3.2, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"7526fae47372\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/7526fae47372.parquet\", \"means_path\": \"data/sim/power/means/7526fae47372.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 0.15, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"80c578ceca0f\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/80c578ceca0f.parquet\", \"means_path\": \"data/sim/power/means/80c578ceca0f.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 0.4, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"231e0b71aa62\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/231e0b71aa62.parquet\", \"means_path\": \"data/sim/power/means/231e0b71aa62.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 0.8, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"07166827ed20\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/07166827ed20.parquet\", \"means_path\": \"data/sim/power/means/07166827ed20.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 1.6, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"ad91a73d676a\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/ad91a73d676a.parquet\", \"means_path\": \"data/sim/power/means/ad91a73d676a.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 3.2, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"86e7d6f37e89\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/86e7d6f37e89.parquet\", \"means_path\": \"data/sim/power/means/86e7d6f37e89.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 0.15, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"67fa152cc2ce\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/67fa152cc2ce.parquet\", \"means_path\": \"data/sim/power/means/67fa152cc2ce.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 0.4, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"221d8c9b256a\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/221d8c9b256a.parquet\", \"means_path\": \"data/sim/power/means/221d8c9b256a.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 0.8, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"e06fefdb38ef\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/e06fefdb38ef.parquet\", \"means_path\": \"data/sim/power/means/e06fefdb38ef.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 1.6, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"d17946a60411\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/d17946a60411.parquet\", \"means_path\": \"data/sim/power/means/d17946a60411.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 3.2, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"7f75c1bbad2c\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/7f75c1bbad2c.parquet\", \"means_path\": \"data/sim/power/means/7f75c1bbad2c.npz\", \"_rank\": 4, \"_sweep\": \"power\"}]")

def _safe(job):
    try:
        return run_cell(job)
    except Exception as e:
        print('[FAIL]', job['config_id'], repr(e))
        traceback.print_exc()
        return job['config_id'], -1.0

t0 = time.time()
results = []
for i, job in enumerate(JOBS):
    if time.time() - t0 > 8.6 * 3600:
        print('[WALL LIMIT] stopping cleanly after', i, 'jobs')
        break
    results.append(_safe(job))
print('shard done:', results)

In [ ]:
import hashlib, json, glob, os
manifest = {'shard_id': 7, 'files': {}}
os.makedirs('data', exist_ok=True)
for f in sorted(glob.glob('data/**/*.parquet', recursive=True)) + \
         sorted(glob.glob('data/**/*.npz', recursive=True)):
    h = hashlib.sha256(open(f, 'rb').read()).hexdigest()
    manifest['files'][f] = h
with open('data/manifest.json', 'w') as fh:
    json.dump(manifest, fh, indent=1)
print(json.dumps(manifest['files'], indent=1))

In [ ]:
import shutil
archive = shutil.make_archive('scf_shard_{:02d}'.format(7), 'zip', 'data')
print('archived:', archive)
output_file = archive
try:
    from google.colab import files
    files.download(output_file)
    print('Downloaded:', output_file)
except Exception as e:
    print('(Not on Colab / download skipped):', e)